# Análisis de textos

---

El *Procesamiento de Lenguaje Natural (NLP)* es una rama de la inteligencia artificial que permite a las computadoras entender, interpretar y generar lenguaje humano. Su objetivo es transformar textos o discursos en datos que una máquina pueda analizar, facilitando tareas como traducción automática, chatbots, análisis de sentimientos o motores de búsqueda. 📖🤖


##  Vectorización de textos

---

La *vectorización de textos* es el proceso de transformar palabras o frases en representaciones numéricas que las computadoras puedan procesar. 📊 En lugar de trabajar con texto plano, lo convertimos en vectores que permiten a los algoritmos de machine learning analizar, comparar y encontrar patrones en el lenguaje. ✨


In [ ]:
#!pip install wordcloud spacy torch transformers

In [ ]:
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud, STOPWORDS
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix

import spacy


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModel

In [ ]:
model_name = "dccuchile/bert-base-spanish-wwm-cased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)

In [ ]:
#spacy.cli.download("es_core_news_sm")
nlp = spacy.load("es_core_news_sm")

In [ ]:
sns.set_style('whitegrid')
sns.set_palette('viridis')

<center>
  <img src="https://i5.walmartimages.com.mx/gr/images/product-images/img_large/00360054263068L.jpg" alt="Regresion Descenso de Gradiente" style="max-width:50%; height:auto;"  width="30%">
</center>

In [ ]:
opiniones=pd.read_csv("https://raw.githubusercontent.com/zyntonyson/bootcamp_ds_da/refs/heads/main/16-ds-analisis-textos/opiniones_garnier.csv")
opiniones

In [ ]:
opiniones.shape, opiniones.Label.value_counts()

In [ ]:
# Longitud de comentarios y distribución
opiniones["len"] = opiniones["Comentario"].str.len()
opiniones["n_words"] = opiniones["Comentario"].apply(lambda x: len(x.split()))
opiniones.groupby("Label")["len"].describe()

In [ ]:
#Cantidad de palabras
opiniones.groupby("Label")["n_words"].describe()

In [ ]:


sns.countplot(data=opiniones, x="Label", order=["positiva","negativa"])
plt.title("Distribución de clases")
plt.show()

sns.boxplot(data=opiniones, x="Label", y="len", order=["positiva","negativa"])
plt.title("Longitud del texto por clase")
plt.show()

sns.boxplot(data=opiniones, x="Label", y="n_words", order=["positiva","negativa"])
plt.title("Longitud del texto por clase")
plt.show()

In [ ]:
stop_es = set(STOPWORDS) | {"de","la","el","los","las","que","y","mi","me","lo","es"}

def nubes_por_clase(df, label):
    texto = " ".join(df.loc[df.Label==label, "Comentario"].astype(str))
    wc = WordCloud(width=800, height=400, stopwords=stop_es, background_color="white").generate(texto)
    plt.figure(figsize=(8,4)); plt.imshow(wc); plt.axis("off"); plt.title(f"WordCloud: {label}"); plt.show()

for lab in ["positiva","negativa"]:
    nubes_por_clase(opiniones, lab)

In [ ]:
nlp = spacy.load("es_core_news_sm")

doc = nlp("Los estudiantes corriendo aprendieron rápidamente")
print([token.lemma_ for token in doc])

In [ ]:
def lematizar(texto):
    doc = nlp(texto.lower())
    # quitamos puntuación, espacios y stopwords del modelo
    lemmas = [t.lemma_ for t in doc if not (t.is_punct or t.is_space or t.is_stop)]
    return " ".join(lemmas)

opiniones["Comentario_lem"] = opiniones["Comentario"].astype(str).apply(lematizar)
opiniones[["Comentario","Comentario_lem"]].head()

In [ ]:
cv = CountVectorizer(ngram_range=(1,2), min_df=1)
X_bow = cv.fit_transform(opiniones["Comentario"])
y = opiniones["Label"].apply(lambda x: int(x=='positiva'))

cv.get_feature_names_out()[:20], X_bow.shape

In [ ]:
tfidf = TfidfVectorizer(ngram_range=(1,2), min_df=1)
X_tfidf = tfidf.fit_transform(opiniones["Comentario"])
X_tfidf.shape

In [ ]:
# 
X_train, X_test, y_train, y_test = train_test_split(X_bow, y, test_size=0.4, random_state=42, stratify=y)

clf = LogisticRegression(max_iter=200, n_jobs=-1)
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)

print(classification_report(y_test, y_pred, digits=3))
print(confusion_matrix(y_test, y_pred))

In [ ]:
# Cambia X_bow por X_tfidf si quieres usar TF-IDF
X_train, X_test, y_train, y_test = train_test_split(X_tfidf, y, test_size=0.4, random_state=42, stratify=y)

clf = LogisticRegression(max_iter=200, n_jobs=-1)
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)

print(classification_report(y_test, y_pred, digits=3))
print(confusion_matrix(y_test, y_pred))

In [ ]:
def get_embbeding(text):
    doc=nlp(text.lower())
    return doc.vector

In [ ]:
X_embbeding=np.stack(opiniones['Comentario'].apply(get_embbeding).values)

In [ ]:
# Cambia X_bow por X_tfidf si quieres usar TF-IDF
X_train, X_test, y_train, y_test = train_test_split(X_embbeding, y, test_size=0.4, random_state=42, stratify=y)

clf = LogisticRegression(max_iter=200, n_jobs=-1)
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)

print(classification_report(y_test, y_pred, digits=3))
print(confusion_matrix(y_test, y_pred))

In [ ]:
def get_BERT_embedding(texto):
    inputs = tokenizer(texto, return_tensors="pt", truncation=True, padding=True, max_length=50)
    with torch.no_grad():
        outputs = model(**inputs)
    # Tomamos la representación de la última capa oculta (batch_size, seq_len, hidden_size)
    embeddings = outputs.last_hidden_state
    # Promediamos sobre la secuencia → vector fijo (hidden_size=768)
    vector = embeddings.mean(dim=1).squeeze().numpy()
    return vector

In [ ]:
X_BERT=np.vstack(opiniones["Comentario"].apply(get_BERT_embedding).values)

In [ ]:
# Cambia X_bow por X_tfidf si quieres usar TF-IDF
X_train, X_test, y_train, y_test = train_test_split(X_BERT, y, test_size=0.4, random_state=42, stratify=y)

clf = LogisticRegression(max_iter=200, n_jobs=-1)
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)

print(classification_report(y_test, y_pred, digits=3))
print(confusion_matrix(y_test, y_pred))